In [7]:
import pandas as pd
import numpy as np

In [10]:
path='bengaluru_house_prices.csv'
df=pd.read_csv("C:\Users\Rozra\OneDrive\Desktop\Benguluru-House-Price-Prediction-main\Benguluru-House-Price-Prediction-main\bengaluru_house_prices.csv");
df.head()

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (2519543933.py, line 2)

In [ ]:
df.shape

: 

In [ ]:
df.groupby("area_type")["area_type"].agg("count")

: 

In [ ]:
df1=df.drop(["area_type","availability","society"],axis=1)
df1.head(10)

: 

In [ ]:
df1.isnull().sum()

: 

In [ ]:
df1["balcony"]=df1["balcony"].fillna(df1["balcony"].mean())
df1["bath"]=df1["bath"].fillna(df1["bath"].mean())

: 

In [ ]:
df1=df1.dropna()

: 

In [ ]:
df1.isnull().sum()

: 

In [ ]:
df1.shape

: 

In [ ]:
df1["BHK"]=df1["size"].apply(lambda x: int(x.split(' ')[0]))
df2=df1.drop(["size"],axis=1)
df2.head()

: 

In [ ]:
df2["BHK"].unique()

: 

In [ ]:
df2[df2.BHK>20]

: 

So we need to find the histogram for total_sqft to find such outliers. So we need to confirm that the values are single numbers

In [ ]:
df2["total_sqft"].unique()

: 

In [ ]:
def is_float(x):
  try:
    float(x)
  except:
    return False
  return True

: 

In [ ]:
df2[~df2["total_sqft"].apply(is_float)]

: 

Some of the values of total_sqft are in ranges and string

In [ ]:
def convert_range_to_float(x):
  tokens=x.split("-")
  if len(tokens)==2:
    return (float(tokens[0])+float(tokens[1]))/2
  try:
    return float(x)
  except:
    return None


: 

In [ ]:
df2["total_sqft"]=df2["total_sqft"].apply(convert_range_to_float)

: 

In [ ]:
df2.iloc[776]

: 

In [ ]:
df2.head(10)

: 

In [ ]:
df2["price_per_sqft"]=(df2["price"]*100000)/df2["total_sqft"]
df2.head()

: 

Removing rows that is not quite correct like a threshold is set here it is 300(specifying the minimum sqft of each room must be greater than threshold if not remove it.)

In [ ]:
df2.location = df2.location.apply(lambda x: x.strip())
location_stats = df2['location'].value_counts(ascending=False)
location_stats

: 

In [ ]:
location_stats_less_than_10 = location_stats[location_stats<=10]
location_stats_less_than_10

: 

In [ ]:
df2.location = df2.location.apply(lambda x: 'other' if x in location_stats_less_than_10 else x)
len(df2.location.unique())


: 

In [ ]:
df2.shape

: 

In [ ]:
df2[~(df2["total_sqft"]/df2["BHK"]>300)]

: 

In [ ]:
df3=df2[(df2["total_sqft"]/df2["BHK"]>300)]
df3.shape

: 

In [ ]:
df3["total_sqft"].describe()

: 

Now removing outliers such that in a particular location if the price_per_sqft is greater than 1 standard deviation or less than 1 standard deviation.

In [ ]:
def remove_price_ps_otlier(df):
  df_new=pd.DataFrame()
  for key,subdf in df.groupby("location"):
    m=np.mean(subdf.price_per_sqft)
    s=np.std(subdf.price_per_sqft)
    df_temp=subdf[(subdf.price_per_sqft>(m-s)) & (subdf.price_per_sqft<=(m+s))]
    df_new=pd.concat([df_new,df_temp],ignore_index=True)
  return df_new


: 

In [ ]:
df4=remove_price_ps_otlier(df3)
df4.shape

: 

In [ ]:
df4[df4["bath"]>10]

: 

Setting a threshold for the number of bathroom. Such that bathroom count can't exceed grater than number of bedrooms+2.

In [ ]:
df5=df4[df4["bath"]<=(df4["BHK"]+2)]

: 

In [ ]:
df5.head()

: 

In [ ]:
def remove_bhk_outliers(df):
    exclude_indices = np.array([])
    for location, location_df in df.groupby('location'):
        bhk_stats = {}
        for bhk, bhk_df in location_df.groupby('BHK'):
            bhk_stats[bhk] = {
                'mean': np.mean(bhk_df.price_per_sqft),
                'std': np.std(bhk_df.price_per_sqft),
                'count': bhk_df.shape[0]
            }
        for bhk, bhk_df in location_df.groupby('BHK'):
            stats = bhk_stats.get(bhk-1)
            if stats and stats['count']>5:
                exclude_indices = np.append(exclude_indices, bhk_df[bhk_df.price_per_sqft<(stats['mean'])].index.values)
    return df.drop(exclude_indices,axis='index')
df6 = remove_bhk_outliers(df5)
df6.shape

: 

Dropping the features that don't have use further.


In [ ]:
df7=df6.drop(["price_per_sqft"],axis=1)

: 

In [ ]:
dummies=pd.get_dummies(df7["location"])
dummies.head()

: 

In [ ]:
df8=pd.concat([df7.drop(["location","balcony"],axis=1),dummies.drop(["other"],axis=1)],axis=1)
df8.head(10)

: 

In [ ]:
X=df8.drop(["price"],axis=1)
Y=df8["price"]

: 

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=50)

: 

In [ ]:
from sklearn.linear_model import LinearRegression
lr=LinearRegression()
lr.fit(X_train,Y_train)

: 

In [ ]:
lr.score(X_train,Y_train)

: 

In [ ]:
lr.score(X_test,Y_test)

: 

In [ ]:
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_val_score

cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=0)

cross_val_score(LinearRegression(), X, Y, cv=cv)

: 

In [ ]:
from sklearn.model_selection import GridSearchCV

from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor

def find_best_model_using_gridsearchcv(X,y):
    algos = {
        'linear_regression' : {
            'model': LinearRegression(),
            'params': {
            }
        },
        'lasso': {
            'model': Lasso(),
            'params': {
                'alpha': [1,2],
                'selection': ['random', 'cyclic']
            }
        },
        'decision_tree': {
            'model': DecisionTreeRegressor(),
            'params': {
                'criterion' : ['mse','friedman_mse'],
                'splitter': ['best','random']
            }
        }
    }
    scores = []
    cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=0)
    for algo_name, config in algos.items():
        gs =  GridSearchCV(config['model'], config['params'], cv=cv, return_train_score=False)
        gs.fit(X,y)
        scores.append({
            'model': algo_name,
            'best_score': gs.best_score_,
            'best_params': gs.best_params_
        })

    return pd.DataFrame(scores,columns=['model','best_score','best_params'])

find_best_model_using_gridsearchcv(X,Y)

: 

In [ ]:
import pickle
with open("bengluru_price_model.pickle","wb") as f:
  pickle.dump(lr,f)

: 

In [ ]:
import json
columns={
    "data_columns":[col.lower() for col in X.columns]
}
with open("columns.json","w") as f:
  f.write(json.dumps(columns))

: 

: 